# Clasificación de tumores con Random Forest
**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Fuente:** UCI Machine Learning Repository / `sklearn.datasets.load_breast_cancer`  
**Problema:** clasificar tumores como malignos o benignos a partir de características numéricas de núcleos celulares.  
**Variable objetivo:** `target`, donde 0 = maligno y 1 = benigno.  
**Observación:** cada fila representa una muestra de tumor.  
**Métrica principal:** F1-score, porque combina precision y recall. Como métrica complementaria se utiliza accuracy.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
print("Librerías cargadas correctamente.")


Librerías cargadas correctamente.


In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
df = X.copy()
df["target"] = y

print("Dimensiones:", df.shape)
print("Observaciones:", len(df))
print("Variables predictoras:", X.shape[1])
print("\nPrimeras 5 observaciones:")
print(df.head().to_string())


Dimensiones: (569, 31)
Observaciones: 569
Variables predictoras: 30

Primeras 5 observaciones:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  mean compactness  mean concavity  mean concave points  mean symmetry  mean fractal dimension  radius error  texture error  perimeter error  area error  smoothness error  compactness error  concavity error  concave points error  symmetry error  fractal dimension error  worst radius  worst texture  worst perimeter  worst area  worst smoothness  worst compactness  worst concavity  worst concave points  worst symmetry  worst fractal dimension  target
0        17.99         10.38          122.80     1001.0          0.11840           0.27760          0.3001              0.14710         0.2419                 0.07871        1.0950         0.9053            8.589      153.40          0.006399            0.04904          0.05373               0.01587         0.03003                 0.006193         25.38          17.33          

In [3]:
print("Valores faltantes:", df.isnull().sum().sum())
print("Duplicados:", df.duplicated().sum())
print("\nDistribución objetivo:")
print(y.value_counts())
print("\nPorcentaje:")
print((y.value_counts(normalize=True)*100).round(2))


Valores faltantes: 0
Duplicados: 0

Distribución objetivo:
target
1    357
0    212

Porcentaje:
target
1    62.74
0    37.26


## Preparación de datos
Se utiliza una división 80/20 con semilla fija 42. Se aplica estratificación para conservar la proporción de clases en entrenamiento y prueba. Todas las variables predictoras son numéricas. Random Forest no requiere escalamiento para funcionar correctamente.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (455, 30)
Prueba: (114, 30)


In [5]:
modelo_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
inicio = time.perf_counter()
modelo_base.fit(X_train, y_train)
tiempo_base = time.perf_counter() - inicio

pred_base = modelo_base.predict(X_test)
f1_base = f1_score(y_test, pred_base)
accuracy_base = accuracy_score(y_test, pred_base)

print("Características:", X_train.shape[1])
print(f"F1-score: {f1_base:.4f}")
print(f"Accuracy: {accuracy_base:.4f}")
print(f"Tiempo: {tiempo_base:.4f} segundos")


Características: 30
F1-score: 0.9655
Accuracy: 0.9561
Tiempo: 0.1798 segundos


## Selección de características
Se utiliza `SelectFromModel` con Random Forest y el umbral de la mediana de las importancias. El método conserva las variables cuya importancia alcanza el criterio definido. La selección se ajusta únicamente con los datos de entrenamiento, evitando fuga de información y manteniendo la interpretación de las variables originales.


In [6]:
selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    threshold="median"
)
selector.fit(X_train, y_train)
X_train_reducido = selector.transform(X_train)
X_test_reducido = selector.transform(X_test)
variables_seleccionadas = X.columns[selector.get_support()].tolist()

print("Características originales:", X_train.shape[1])
print("Características seleccionadas:", len(variables_seleccionadas))
print("\nVariables conservadas:")
for v in variables_seleccionadas:
    print("-", v)


Características originales: 30
Características seleccionadas: 15

Variables conservadas:
- mean radius
- mean perimeter
- mean area
- mean concavity
- mean concave points
- radius error
- area error
- worst radius
- worst texture
- worst perimeter
- worst area
- worst smoothness
- worst compactness
- worst concavity
- worst concave points


In [7]:
modelo_reducido = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
inicio = time.perf_counter()
modelo_reducido.fit(X_train_reducido, y_train)
tiempo_reducido = time.perf_counter() - inicio

pred_reducido = modelo_reducido.predict(X_test_reducido)
f1_reducido = f1_score(y_test, pred_reducido)
accuracy_reducido = accuracy_score(y_test, pred_reducido)

print(f"F1-score: {f1_reducido:.4f}")
print(f"Accuracy: {accuracy_reducido:.4f}")
print(f"Tiempo: {tiempo_reducido:.4f} segundos")


F1-score: 0.9655
Accuracy: 0.9561
Tiempo: 0.1613 segundos


In [8]:
comparacion_reduccion = pd.DataFrame({
    "Configuración": ["Modelo base", "Modelo con selección"],
    "Número de características": [X_train.shape[1], X_train_reducido.shape[1]],
    "F1-score": [f1_base, f1_reducido],
    "Accuracy": [accuracy_base, accuracy_reducido],
    "Tiempo de entrenamiento (s)": [tiempo_base, tiempo_reducido]
})
print(comparacion_reduccion.round(4).to_string(index=False))


       Configuración  Número de características  F1-score  Accuracy  Tiempo de entrenamiento (s)
         Modelo base                         30    0.9655    0.9561                       0.1798
Modelo con selección                         15    0.9655    0.9561                       0.1613


## Optimización de hiperparámetros
Se aplica `GridSearchCV` con validación cruzada estratificada de 5 particiones y F1-score como criterio. La validación cruzada permite evaluar cada alternativa en varios subconjuntos, ofreciendo una comparación más estable que una sola división. El conjunto de prueba se reserva exclusivamente para la evaluación final, evitando utilizarlo para tomar decisiones de ajuste.


In [9]:
pipeline_optimizado = Pipeline([
    ("selector", SelectFromModel(
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
        threshold="median"
    )),
    ("modelo", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [None, 5, 10],
    "modelo__min_samples_split": [2, 5],
    "modelo__max_features": ["sqrt", "log2"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipeline_optimizado, param_grid, scoring="f1", cv=cv, n_jobs=-1)

inicio = time.perf_counter()
grid.fit(X_train, y_train)
tiempo_optimizado = time.perf_counter() - inicio

print("Mejores hiperparámetros:", grid.best_params_)
print("Mejor F1 CV:", round(grid.best_score_, 4))
print("Tiempo de búsqueda:", round(tiempo_optimizado, 4), "segundos")


Mejores hiperparámetros: {'modelo__max_depth': 5, 'modelo__max_features': 'sqrt', 'modelo__min_samples_split': 2, 'modelo__n_estimators': 200}
Mejor F1 CV: 0.9650
Tiempo de búsqueda: 43.5406 segundos


In [10]:
tabla_hiperparametros = pd.DataFrame({
    "Hiperparámetro": ["n_estimators", "max_depth", "min_samples_split", "max_features"],
    "Valores explorados": ["100, 200", "None, 5, 10", "2, 5", "sqrt, log2"],
    "Mejor valor": [
        grid.best_params_["modelo__n_estimators"],
        grid.best_params_["modelo__max_depth"],
        grid.best_params_["modelo__min_samples_split"],
        grid.best_params_["modelo__max_features"]
    ]
})
print(tabla_hiperparametros.to_string(index=False))


   Hiperparámetro Valores explorados Mejor valor
     n_estimators           100, 200         200
        max_depth        None, 5, 10           5
min_samples_split               2, 5           2
     max_features         sqrt, log2        sqrt


In [11]:
modelo_optimizado = grid.best_estimator_
pred_optimizado = modelo_optimizado.predict(X_test)
f1_optimizado = f1_score(y_test, pred_optimizado)
accuracy_optimizado = accuracy_score(y_test, pred_optimizado)

selector_final = modelo_optimizado.named_steps["selector"]
variables_finales = X.columns[selector_final.get_support()].tolist()

print(f"F1-score: {f1_optimizado:.4f}")
print(f"Accuracy: {accuracy_optimizado:.4f}")
print("Características:", len(variables_finales))
print("\nReporte de clasificación:")
print(classification_report(y_test, pred_optimizado, target_names=["Maligno", "Benigno"]))


F1-score: 0.9655
Accuracy: 0.9561
Características: 15

Reporte de clasificación:
              precision    recall  f1-score   support

     Maligno       0.95      0.93      0.94        42
     Benigno       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114


In [12]:
tabla_final = pd.DataFrame({
    "Modelo": ["Base", "Reducido", "Optimizado"],
    "F1-score": [f1_base, f1_reducido, f1_optimizado],
    "Accuracy": [accuracy_base, accuracy_reducido, accuracy_optimizado],
    "Características": [X_train.shape[1], X_train_reducido.shape[1], len(variables_finales)],
    "Tiempo (s)": [tiempo_base, tiempo_reducido, tiempo_optimizado]
})
print(tabla_final.round(4).to_string(index=False))


    Modelo  F1-score  Accuracy  Características  Tiempo (s)
      Base    0.9655    0.9561               30      0.1798
  Reducido    0.9655    0.9561               15      0.1613
Optimizado    0.9655    0.9561               15     43.5406


In [13]:
cm = confusion_matrix(y_test, pred_optimizado)
disp = ConfusionMatrixDisplay(cm, display_labels=["Maligno", "Benigno"])
disp.plot()
plt.title("Matriz de confusión - Modelo optimizado")
plt.show()

print("Matriz de confusión:")
print(cm)


Matriz de confusión:
[[39  3]
 [ 2 70]]


## Conclusión
El modelo base alcanzó un F1-score de **0.9655** y el modelo reducido obtuvo **0.9655**, utilizando 15 características en lugar de 30. El modelo optimizado obtuvo un F1-score de **0.9655** y accuracy de **0.9561**. En esta ejecución, la configuración con mayor F1-score fue **Base**.

La selección de características permitió disminuir la complejidad del conjunto de datos manteniendo un desempeño comparable. La optimización mediante GridSearchCV implicó un mayor costo computacional porque evaluó múltiples combinaciones mediante validación cruzada. Por ello, la decisión final debe considerar no solo una diferencia pequeña en desempeño, sino también la cantidad de variables, interpretabilidad y tiempo requerido.

Como limitación, el análisis utiliza un conjunto relativamente pequeño y una sola partición final de entrenamiento/prueba. Como trabajo posterior, sería conveniente aplicar validación cruzada repetida, comparar otros métodos de selección y evaluar otros algoritmos de ensamble.
